# 使用 LangChain、Deep Lake 和 OpenAI 实现问答

本 notebook 展示如何使用 LangChain、作为向量存储的 [Deep Lake](https://activeloop.ai/) 以及 OpenAI 嵌入来实现一个问答系统。我们将按以下步骤完成：

1. 加载一个 Deep Lake 文本数据集
2. 初始化一个 [基于 LangChain 的 Deep Lake 向量存储](https://docs.activeloop.ai/tutorials/vector-store/deep-lake-vector-store-in-langchain)
3. 向向量存储中添加文本
4. 在数据库上运行查询
5. 完成！

你还可以参考其他教程，例如针对任意类型数据（PDF、json、csv、文本）的问答：[与存储在 Deep Lake 中的任意数据对话](https://www.activeloop.ai/resources/data-chad-an-ai-app-with-lang-chain-deep-lake-to-chat-with-any-data/)、[代码理解](https://www.activeloop.ai/resources/lang-chain-gpt-4-for-code-understanding-twitter-algorithm/)、[基于 PDF 的问答](https://www.activeloop.ai/resources/ultimate-guide-to-lang-chain-deep-lake-build-chat-gpt-to-answer-questions-on-your-financial-data/)，或[歌曲推荐](https://www.activeloop.ai/resources/3-ways-to-build-a-recommendation-engine-for-songs-with-lang-chain/)。

## 安装依赖
我们来安装以下软件包。

In [ ]:
!pip install deeplake langchain openai tiktoken

## 身份验证
在这里提供你的 OpenAI API 密钥：

In [2]:
import getpass
import os

os.environ['OPENAI_API_KEY'] = getpass.getpass()

··········


## 加载 Deep Lake 文本数据集
本示例将使用 [cohere-wikipedia-22](https://app.activeloop.ai/davitbun/cohere-wikipedia-22) 数据集中的一个 20000 条样本的子集。

In [3]:
import deeplake

ds = deeplake.load("hub://activeloop/cohere-wikipedia-22-sample")
ds.summary()

\

Opening dataset in read-only mode as you don't have write permissions.


-

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/activeloop/cohere-wikipedia-22-sample



|

hub://activeloop/cohere-wikipedia-22-sample loaded successfully.

Dataset(path='hub://activeloop/cohere-wikipedia-22-sample', read_only=True, tensors=['ids', 'metadata', 'text'])

  tensor    htype     shape      dtype  compression
 -------   -------   -------    -------  ------- 
   ids      text    (20000, 1)    str     None   
 metadata   json    (20000, 1)    str     None   
   text     text    (20000, 1)    str     None   


我们来看几个样本：

In [4]:
ds[:3].text.data()["value"]

['The 24-hour clock is a way of telling the time in which the day runs from midnight to midnight and is divided into 24 hours, numbered from 0 to 23. It does not use a.m. or p.m. This system is also referred to (only in the US and the English speaking parts of Canada) as military time or (only in the United Kingdom and now very rarely) as continental time. In some parts of the world, it is called railway time. Also, the international standard notation of time (ISO 8601) is based on this format.',
 'A time in the 24-hour clock is written in the form hours:minutes (for example, 01:23), or hours:minutes:seconds (01:23:45). Numbers under 10 have a zero in front (called a leading zero); e.g. 09:07. Under the 24-hour clock system, the day begins at midnight, 00:00, and the last minute of the day begins at 23:59 and ends at 24:00, which is identical to 00:00 of the following day. 12:00 can only be mid-day. Midnight is called 24:00 and is used to mean the end of the day and 00:00 is used to me

## LangChain 的 Deep Lake 向量存储
我们来定义一个 `dataset_path`，你的 Deep Lake 向量存储将在这里存放文本嵌入。

In [5]:
dataset_path = 'wikipedia-embeddings-deeplake'

我们将把 OpenAI 的 `text-embedding-3-small` 设置为嵌入函数，并在 `dataset_path` 处初始化一个 Deep Lake 向量存储……

In [6]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import DeepLake

embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = DeepLake(dataset_path, embedding=embedding, overwrite=True)

……然后使用 `add_texts` 方法，逐批向其中填充样本。

In [7]:
from tqdm.auto import tqdm

batch_size = 100

nsamples = 10  # for testing. Replace with len(ds) to append everything
for i in tqdm(range(0, nsamples, batch_size)):
    # find end of batch
    i_end = min(nsamples, i + batch_size)

    batch = ds[i:i_end]
    id_batch = batch.ids.data()["value"]
    text_batch = batch.text.data()["value"]
    meta_batch = batch.metadata.data()["value"]

    db.add_texts(text_batch, metadatas=meta_batch, ids=id_batch)

  0%|          | 0/1 [00:00<?, ?it/s]


creating embeddings: 100%|██████████| 1/1 [00:02<00:00,  2.11s/it]

100%|██████████| 10/10 [00:00<00:00, 462.42it/s]

Dataset(path='wikipedia-embeddings-deeplake', tensors=['text', 'metadata', 'embedding', 'id'])

  tensor      htype      shape      dtype  compression
  -------    -------    -------    -------  ------- 
   text       text      (10, 1)      str     None   
 metadata     json      (10, 1)      str     None   
 embedding  embedding  (10, 1536)  float32   None   
    id        text      (10, 1)      str     None   


## 在数据库上运行用户查询
底层的 Deep Lake 数据集对象可以通过 `db.vectorstore.dataset` 访问，其数据结构可以用 `db.vectorstore.summary()` 进行概览，结果显示有 4 个张量、10 个样本：

In [8]:
db.vectorstore.summary()

Dataset(path='wikipedia-embeddings-deeplake', tensors=['text', 'metadata', 'embedding', 'id'])

  tensor      htype      shape      dtype  compression
  -------    -------    -------    -------  ------- 
   text       text      (10, 1)      str     None   
 metadata     json      (10, 1)      str     None   
 embedding  embedding  (10, 1536)  float32   None   
    id        text      (10, 1)      str     None   


现在我们将在向量存储上搭建问答，并使用 GPT-3.5-Turbo 作为 LLM。

In [9]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

# Re-load the vector store in case it's no longer initialized
# db = DeepLake(dataset_path = dataset_path, embedding_function=embedding)

qa = RetrievalQA.from_chain_type(llm=ChatOpenAI(model='gpt-3.5-turbo'), chain_type="stuff", retriever=db.as_retriever())

我们来运行一个提示词并查看输出。在内部，这个 API 会执行嵌入搜索，找出最相关的数据并送入 LLM 的上下文。

In [10]:
query = 'Why does the military not say 24:00?'
qa.run(query)

'The military prefers not to say 24:00 because they do not like to have two names for the same thing. Instead, they always say "23:59", which is one minute before midnight.'

大功告成！